# Advanced Retrieval Methods: Comparative Evaluation

## Objective

This notebook provides a comprehensive evaluation of various retrieval strategies using the LangChain framework. We compare six different retriever implementations across two chunking strategies:

**Retriever Methods:**
1. Naive Retrieval (Baseline)
2. BM25 Retrieval
3. Contextual Compression (Reranking)
4. Multi-Query Retrieval
5. Parent Document Retrieval
6. Ensemble Retrieval

**Chunking Strategies:**
1. Standard Chunking (Fixed-size with RecursiveCharacterTextSplitter)
2. Semantic Chunking (Content-aware boundary detection)

## Evaluation Methodology

We use Ragas (Retrieval Augmented Generation Assessment) framework to:
- Generate synthetic test datasets with ground truth
- Measure retriever performance using standard metrics:
  - Context Precision
  - Context Recall
  - Context Relevancy
- Track cost and latency via LangSmith

## Dataset

The evaluation uses a corpus of 50 AI/ML project descriptions from the AIE bootcamp, including project metadata such as domains, scores, and judge feedback.


---

## Section 1: Environment Setup and Dependencies


In [1]:
# Core Python libraries
import os
import pandas as pd
from datetime import datetime
from typing import List, Dict, Any
from dotenv import load_dotenv

# Load environment variables
load_dotenv()

# Verify required API keys
required_keys = ["OPENAI_API_KEY", "COHERE_API_KEY", "LANGCHAIN_API_KEY"]
missing_keys = [key for key in required_keys if not os.getenv(key)]

if missing_keys:
    raise ValueError(f"Missing API keys: {', '.join(missing_keys)}")

print(f"API keys loaded | LangSmith tracing: {os.getenv('LANGCHAIN_TRACING_V2', 'false')}")


API keys loaded | LangSmith tracing: true


### Import Dependencies


In [2]:
# LangChain Core
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter

# LangChain Document Loaders and Text Splitters
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_experimental.text_splitter import SemanticChunker

# LangChain Models
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# LangChain Vector Stores
from langchain_community.vectorstores import Qdrant
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

# LangChain Retrievers
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain.storage import InMemoryStore

# Cohere for Reranking
from langchain_cohere import CohereRerank

# Ragas for Evaluation (v0.2.10)
from ragas.testset import TestsetGenerator
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas import evaluate

print("All dependencies imported successfully")


All dependencies imported successfully


### Initialize Models and Embeddings


In [3]:
# Initialize models
chat_model = ChatOpenAI(model="gpt-4.1-nano", temperature=0)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

print(f"Models initialized: {chat_model.model_name} | {embeddings.model}")


Models initialized: gpt-4.1-nano | text-embedding-3-small


---

## Implementation Plan

### Overview

This evaluation compares **12 retriever configurations**:
- 6 retriever methods × 2 chunking strategies = 12 total evaluations

### Execution Phases

#### Phase 1: Setup and Data Preparation
1. **Environment Configuration**
   - Load dependencies (LangChain, Ragas, Cohere, Qdrant)
   - Load API keys from .env file (OpenAI, Cohere, LangSmith)
   - Enable LangSmith tracing for cost/latency tracking

2. **Data Loading**
   - Load "howpeopleuseai.pdf" from data folder
   - Convert to LangChain Document format using PyPDFLoader
   - Verify document structure and content

3. **Document Chunking**
   - Create standard chunks using RecursiveCharacterTextSplitter
   - Create semantic chunks using SemanticChunker
   - Compare chunk statistics

4. **Vector Store Creation**
   - Build Qdrant vector stores for both chunking strategies
   - Configure embeddings (OpenAI text-embedding-3-small)
   - Verify vector store operations

#### Phase 2: Golden Dataset Generation
5. **Synthetic Test Generation**
   - Use Ragas TestsetGenerator with document-based approach
   - Generate 10 test questions with ground truth
   - Include diverse question types (simple, reasoning, multi-context)
   - Validate generated questions

#### Phase 3: Retriever Implementation
6. **Standard Chunking Retrievers**
   - Implement all 6 retrievers using standard chunks
   - Configure parameters to match baseline notebook

7. **Semantic Chunking Retrievers**
   - Implement all 6 retrievers using semantic chunks
   - Maintain consistent configuration

#### Phase 4: Evaluation Execution
8. **Run Evaluations**
   - Execute each of 12 configurations against golden dataset (15 questions)
   - Collect retriever-appropriate Ragas metrics per assignment requirements
   - Track execution time per retriever

9. **Cost and Latency Analysis**
   - Extract metrics from LangSmith traces
   - Calculate average cost per query
   - Measure average latency per retriever

#### Phase 5: Analysis and Reporting
10. **Comparative Analysis**
    - Create results comparison table
    - Analyze chunking strategy impact
    - Identify best-performing retrievers
    
11. **Final Recommendations**
    - Synthesize findings
    - Recommend optimal configuration
    - Discuss tradeoffs (performance vs cost vs latency)

### Expected Outputs

1. Comprehensive metrics table
2. Chunking strategy comparison
3. Cost-benefit analysis
4. Performance recommendations
5. Implementation insights


---

## Section 2: Data Loading


In [4]:
# Load all PDF documents from data folder
loader = DirectoryLoader("./data/", glob="*.pdf", loader_cls=PyMuPDFLoader)
documents = loader.load()

print(f"Loaded {len(documents)} pages from data folder")


Loaded 64 pages from data folder


---

## Section 3: Golden Dataset Generation

Generate synthetic test questions using Ragas TestsetGenerator. This step creates the evaluation dataset before building retrieval infrastructure.


In [5]:
# Configuration
CACHE_FILE = './golden_dataset_cache.csv'
FORCE_REGENERATE = False  # Set to True to regenerate golden dataset

# Check for cached dataset
if os.path.exists(CACHE_FILE) and not FORCE_REGENERATE:
    print(f"Loading cached golden dataset from {CACHE_FILE}")
    golden_dataset = pd.read_csv(CACHE_FILE)
    print(f"Loaded {len(golden_dataset)} questions from cache")
else:
    print("Generating new golden dataset using Ragas...")
    
    # Generate synthetic test dataset using Ragas v0.2.10
    generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
    generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())
    
    generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
    
    # Generate 10 test questions
    testset = generator.generate_with_langchain_docs(
        documents,
        testset_size=10
    )
    
    # Convert to DataFrame
    golden_dataset = testset.to_pandas()
    
    # Cache for future runs
    golden_dataset.to_csv(CACHE_FILE, index=False)
    print(f"Generated and cached {len(golden_dataset)} questions to {CACHE_FILE}")

# Display dataset
golden_dataset


Loading cached golden dataset from ./golden_dataset_cache.csv
Loaded 12 questions from cache


,user_input,reference_contexts,reference,synthesizer_name
0,when was November 2022,['Introduction ChatGPT launched in November 20...,Introduction ChatGPT launched in November 2022.,single_hop_specifc_query_synthesizer
1,Considering the detailed data on ChatGPT's usa...,['Table 1: ChatGPT daily message counts (milli...,"According to the provided context, nearly 80% ...",single_hop_specifc_query_synthesizer
2,How do different occupations use ChatGPT?,['Variation by Occupation Figure 23 presents v...,Variation by Occupation Figure 23 presents var...,single_hop_specifc_query_synthesizer
3,How does the concept of management relate to t...,['Conclusion This paper studies the rapid grow...,The context discusses ChatGPT's rapid growth a...,single_hop_specifc_query_synthesizer
4,Considering the rapid adoption of ChatGPT sinc...,['<1-hop>\n\nIntroduction ChatGPT launched in ...,The context indicates that since ChatGPT's lau...,multi_hop_abstract_query_synthesizer
5,chatbot messages types asking doing expressing...,['<1-hop>\n\nIntroduction ChatGPT launched in ...,The context explains that messages sent to Cha...,multi_hop_abstract_query_synthesizer
6,how AI apps like ChatGPT can be used in work a...,['<1-hop>\n\nIntroduction ChatGPT launched in ...,"The context explains that ChatGPT, launched in...",multi_hop_abstract_query_synthesizer
7,How does the increasing share of non-work mess...,['<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (...,The data shows that from June 2024 to June 202...,multi_hop_abstract_query_synthesizer
8,OpenAI ChatGPT how use for learn and work and ...,['<1-hop>\n\nTable 1: ChatGPT daily message co...,"Based on the context, OpenAI's ChatGPT is wide...",multi_hop_specific_query_synthesizer
9,How does OpenAI's development and widespread a...,['<1-hop>\n\nTable 1: ChatGPT daily message co...,The context shows that OpenAI launched ChatGPT...,multi_hop_specific_query_synthesizer


In [6]:
# Cache Management - Uncomment to clear caches and regenerate

# Clear golden dataset cache
# if os.path.exists('./golden_dataset_cache.csv'):
#     os.remove('./golden_dataset_cache.csv')
#     print("✓ Golden dataset cache cleared")

# Clear evaluation caches
# import shutil
# if os.path.exists('./eval_cache'):
#     shutil.rmtree('./eval_cache')
#     print("✓ All evaluation caches cleared")


---

## Section 3: Document Chunking

We create two sets of chunks from the same source documents to compare retrieval performance:

1. **Standard Chunking**: Fixed-size chunks using RecursiveCharacterTextSplitter
2. **Semantic Chunking**: Content-aware chunks using SemanticChunker


### Strategy 1: Standard Chunking


In [7]:
# Create token-based text splitter
import tiktoken
import time

encoder = tiktoken.encoding_for_model("gpt-4.1-nano")

standard_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,  # tokens
    chunk_overlap=50,  # tokens
    length_function=lambda text: len(encoder.encode(text))
)

start_time = time.time()
standard_chunks = standard_splitter.split_documents(documents)
standard_chunk_time = time.time() - start_time

print(f"Standard chunks: {len(standard_chunks)} (avg: {sum(len(encoder.encode(c.page_content)) for c in standard_chunks) / len(standard_chunks):.0f} tokens, {standard_chunk_time:.2f}s)")


Standard chunks: 87 (avg: 314 tokens, 0.04s)


### Strategy 2: Semantic Chunking


In [8]:
# Create semantic chunker
semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

start_time = time.time()
semantic_chunks = semantic_chunker.split_documents(documents)
semantic_chunk_time = time.time() - start_time

print(f"Semantic chunks: {len(semantic_chunks)} (avg: {sum(len(c.page_content) for c in semantic_chunks) / len(semantic_chunks):.0f} chars, {semantic_chunk_time:.2f}s)")


Semantic chunks: 133 (avg: 848 chars, 25.71s)


### Chunking Strategy Comparison


In [9]:
# Comparison statistics
comparison_df = pd.DataFrame({
    'Metric': ['Total Chunks', 'Avg Size', 'Min Size', 'Max Size', 'Time (s)'],
    'Standard (tokens)': [
        len(standard_chunks),
        int(sum(len(encoder.encode(c.page_content)) for c in standard_chunks) / len(standard_chunks)),
        min(len(encoder.encode(c.page_content)) for c in standard_chunks),
        max(len(encoder.encode(c.page_content)) for c in standard_chunks),
        f"{standard_chunk_time:.2f}"
    ],
    'Semantic (chars)': [
        len(semantic_chunks),
        int(sum(len(c.page_content) for c in semantic_chunks) / len(semantic_chunks)),
        min(len(c.page_content) for c in semantic_chunks),
        max(len(c.page_content) for c in semantic_chunks),
        f"{semantic_chunk_time:.2f}"
    ]
})

print("\nChunking Comparison:")
print(comparison_df.to_string(index=False))



Chunking Comparison:
      Metric Standard (tokens) Semantic (chars)
Total Chunks                87              133
    Avg Size               313              847
    Min Size                29                1
    Max Size               493             3854
    Time (s)              0.04            25.71


---

## Section 4: Vector Store Creation

Creating Qdrant vector stores for both chunking strategies. These will serve as the foundation for all retriever implementations.


### Vector Store 1: Standard Chunks


In [10]:
# Create Qdrant vector store with standard chunks
vectorstore_standard = Qdrant.from_documents(
    standard_chunks,
    embeddings,
    location=":memory:",
    collection_name="standard_chunks"
)

print(f"Standard vector store created: {len(standard_chunks)} documents")


Standard vector store created: 87 documents


### Vector Store 2: Semantic Chunks


In [11]:
# Create Qdrant vector store with semantic chunks
vectorstore_semantic = Qdrant.from_documents(
    semantic_chunks,
    embeddings,
    location=":memory:",
    collection_name="semantic_chunks"
)

print(f"Semantic vector store created: {len(semantic_chunks)} documents indexed")

Semantic vector store created: 133 documents indexed


---

## Section 6: Retriever Implementation - Standard Chunking

Implementing all 6 retriever methods using standard (token-based) chunks.


### Evaluation Utility Functions


In [12]:
# Import evaluation utilities with automatic LangSmith cost tracking
from evaluation_utils import evaluate_retriever_config

print("Evaluation utilities loaded")


Evaluation utilities loaded


#### Record Naive Retriever Metrics


In [ ]:
# Create naive retrievers
naive_retriever_standard = vectorstore_standard.as_retriever(search_kwargs={"k": 10})
naive_retriever_semantic = vectorstore_semantic.as_retriever(search_kwargs={"k": 10})

# Evaluate standard chunking
naive_standard_metrics = evaluate_retriever_config(
    naive_retriever_standard, 
    "Naive", 
    "Standard", 
    golden_dataset,
    llm=chat_model,
    embeddings=embeddings
)

# Evaluate semantic chunking
naive_semantic_metrics = evaluate_retriever_config(
    naive_retriever_semantic, 
    "Naive", 
    "Semantic", 
    golden_dataset,
    llm=chat_model,
    embeddings=embeddings
)



#### Record BM25 Retriever Metrics 


In [ ]:
# Create BM25 retrievers
bm25_retriever_standard = BM25Retriever.from_documents(standard_chunks, k=10)
bm25_retriever_semantic = BM25Retriever.from_documents(semantic_chunks, k=10)

# Evaluate both
bm25_standard_metrics = evaluate_retriever_config(
    bm25_retriever_standard, 
    "BM25", 
    "Standard", 
    golden_dataset,
    llm=chat_model,
    embeddings=embeddings
)

bm25_semantic_metrics = evaluate_retriever_config(
    bm25_retriever_semantic, 
    "BM25", 
    "Semantic", 
    golden_dataset,
    llm=chat_model,
    embeddings=embeddings
)
